# Lab 2: Build the Pipeline

**AI-2: AI Backend Engineering** | Northbrook Partners Document QA

---

You've explored each piece of the AI-2 pipeline in class. Now you'll wire them together yourself.

**Scenario:** Given 8 Northbrook Partners documents and 5 evaluation questions, build an ingestion and Q&A pipeline from the modules you've been using.

**What you will build:**
1. A knowledge base builder — chunk documents, attach metadata, ingest into ChromaDB
2. An ask function — retrieve relevant chunks and generate answers via Claude
3. An evaluation loop — test your pipeline against expected facts

**Rubric:**

| Section | Points |
|---------|--------|
| `build_knowledge_base()` | 25 |
| `ask()` | 35 |
| `evaluate()` | 25 |
| Analysis (Q1 + Q2) | 15 |
| **Total** | **100** |

**Passing: 70+ points.** Partial credit for partially correct functions.

**Estimated time: 2–3 hours**

In [ ]:
import sys, json
from pathlib import Path
from dotenv import load_dotenv

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
load_dotenv()

from src.s0_generation.generate import call_claude_with_usage
from src.s3_ingestion.chunker import chunk_document, count_tokens, Chunk
from src.s3_ingestion.store import ingest_chunks, verify_ingestion, reset_collection, CHROMA_PATH
import chromadb

print("Setup complete.")

In [ ]:
LAB_COLLECTION = "lab2"
DATA_DIR = _root / "data" / "northbrook"

DOC_FILES = [
    "remote_work_policy.md",
    "memo_office_relocation.md",
    "expense_policy.md",
    "memo_ceo_annual_kickoff.md",
    "employee_handbook.md",
    "memo_security_update.md",
    "financial_report_q4_2024.md",
    "engineering_standup_2025_01.md",
]

docs = {}
for f in DOC_FILES:
    docs[f] = (DATA_DIR / f).read_text()
print(f"Loaded {len(docs)} documents ({sum(count_tokens(t) for t in docs.values()):,} total tokens)")

## Section 1: Build the Knowledge Base

**Skills tested:** Chunking (Session 2.2), Metadata (Session 1.2)

Your job: write a function that takes raw documents, chunks them, and ingests them into ChromaDB. The function should return statistics about what was ingested.

**Functions you'll use:**
- `chunk_document(text, source, chunk_size, overlap)` → list of `Chunk` objects
- `reset_collection(collection_name=...)` → clears existing data
- `ingest_chunks(chunks, collection_name=...)` → loads chunks into ChromaDB

Each `Chunk` has a `.text` and `.metadata` dict with keys: `source`, `chunk_index`, `token_count`, `doc_type`.

In [ ]:
def build_knowledge_base(docs: dict[str, str], collection_name: str,
                         chunk_size: int = 128, overlap: int = 25) -> dict:
    """Chunk all documents, add metadata, and ingest into ChromaDB.

    Args:
        docs: {"filename.md": "full text content", ...}
        collection_name: ChromaDB collection name
        chunk_size: tokens per chunk
        overlap: token overlap between chunks

    Returns:
        dict with keys: "total_chunks", "sources" (dict of source -> chunk count),
                        "avg_tokens" (average token count per chunk)
    """
    all_chunks = []

    # -- Step 1 -------------------------------------------------------
    # Loop through each document in `docs`.
    # For each one, call chunk_document() to split it into chunks.
    # Hint: chunk_document(text, source=filename, chunk_size=..., overlap=...)
    # Collect all chunks into all_chunks.

    # YOUR CODE HERE


    # -- Step 2 -------------------------------------------------------
    # Reset the collection (for idempotent re-runs), then ingest.
    # Hint: reset_collection(collection_name=...)
    #        ingest_chunks(all_chunks, collection_name=...)

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Build and return the stats dict.
    # Calculate: total chunks, chunks per source, average token count.
    # Hint: each chunk has chunk.metadata["source"] and chunk.metadata["token_count"]

    # YOUR CODE HERE

    return {"total_chunks": ..., "sources": ..., "avg_tokens": ...}

In [ ]:
# -- Test your function ------------------------------------------------
stats = build_knowledge_base(docs, LAB_COLLECTION, chunk_size=128, overlap=25)
print(f"Total chunks: {stats['total_chunks']}")
print(f"Avg tokens/chunk: {stats['avg_tokens']:.1f}")
for src, count in sorted(stats['sources'].items()):
    print(f"  {src}: {count} chunks")

In [ ]:
# -- Verification: does retrieval work? --------------------------------
test = verify_ingestion("remote work policy", n_results=3, collection_name=LAB_COLLECTION)
for i, doc in enumerate(test["documents"][0]):
    src = test["metadatas"][0][i]["source"]
    score = round(1 - test["distances"][0][i], 3)
    print(f"  [{score}] {src}: {doc[:80]}...")

## Section 2: Build the Ask Function

**Skills tested:** API calls (Session 1.1), RAG retrieval (Session 2.2)

Your job: write a function that retrieves relevant chunks from ChromaDB, builds a prompt with context, and calls Claude to generate an answer.

**Functions you'll use:**
- `verify_ingestion(query, n_results, collection_name)` → ChromaDB query results
- `call_claude_with_usage(prompt, system_prompt, temperature, model)` → dict with `text`, `input_tokens`, `output_tokens`

**Key detail:** ChromaDB returns *distances* (lower = more similar). Convert to similarity scores: `similarity = 1 - distance`.

In [ ]:
SYSTEM_PROMPT = (
    "You are a Q&A assistant for Northbrook Partners. "
    "Answer using ONLY the provided context. "
    "Cite your sources by name. "
    "If the context is insufficient, say: "
    "'I don't have enough information to answer that question.'"
)

In [ ]:
def ask(question: str, collection_name: str = "lab1",
        top_k: int = 3, system_prompt: str = SYSTEM_PROMPT) -> dict:
    """Retrieve relevant chunks from ChromaDB and generate an answer using Claude.

    Args:
        question: the user's question
        collection_name: which ChromaDB collection to search
        top_k: number of chunks to retrieve
        system_prompt: system prompt for Claude

    Returns:
        dict with keys: "answer" (str), "sources" (list of dicts with
        "text", "source", "score"), "input_tokens" (int)
    """
    # -- Step 1 -------------------------------------------------------
    # Query ChromaDB for the top_k most relevant chunks.
    # Hint: verify_ingestion(question, n_results=top_k, collection_name=...)
    # The result has: ["documents"][0], ["metadatas"][0], ["distances"][0]

    # YOUR CODE HERE


    # -- Step 2 -------------------------------------------------------
    # Build a list of source dicts from the results.
    # For each result, create: {"text": ..., "source": metadata["source"],
    #                           "score": round(1 - distance, 4)}
    # ChromaDB returns distances -- convert to similarity: similarity = 1 - distance

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Build the context string for Claude.
    # Format each chunk as: [Source: filename, Score: 0.xxx]\nchunk text
    # Join them with "\n\n---\n\n" between chunks.
    # Then build: f"Context:\n\n{context}\n\n---\n\nQuestion: {question}"

    # YOUR CODE HERE


    # -- Step 4 -------------------------------------------------------
    # Call Claude with the context + question.
    # Hint: call_claude_with_usage(prompt=user_msg, system_prompt=system_prompt,
    #                              temperature=0.0, model="claude-haiku-4-5")

    # YOUR CODE HERE


    # -- Step 5 -------------------------------------------------------
    # Return the structured result.

    return {"answer": ..., "sources": ..., "input_tokens": ...}

In [ ]:
# -- Test your function ------------------------------------------------
result = ask("What equipment support does Northbrook provide for remote workers?")
print(f"Answer: {result['answer'][:300]}...")
print(f"\nSources ({len(result['sources'])}):")
for s in result['sources']:
    print(f"  [{s['score']:.3f}] {s['source']}")
print(f"\nInput tokens: {result['input_tokens']}")

## Section 3: Evaluate Your Pipeline

**Skills tested:** Evaluation thinking (Session 2.1)

You have 5 test questions, each with expected facts the answer should contain. Write a function that runs each question through `ask()` and checks how many expected facts appear in the answer.

This is the same pattern you'll use in Lab 2 to compare naive vs. metadata-aware RAG.

In [ ]:
EVAL_QUESTIONS = {
    "remote_equipment": {
        "question": "What equipment support does Northbrook provide for remote workers?",
        "expected_facts": {
            "one_time_stipend": ["$1,500", "1,500", "1500"],
            "annual_refresh": ["$300", "annual", "each January"],
            "vpn_required": ["VPN", "Cloudflare Zero Trust"],
            "screen_lock": ["5 minutes", "screen lock"],
        },
    },
    "office_relocation": {
        "question": "When is the office relocation and what does the new space include?",
        "expected_facts": {
            "new_address": ["250 Innovation Drive", "Innovation Drive"],
            "move_date": ["March 3", "March 1-2"],
            "size_increase": ["45,000", "40%"],
            "focus_rooms": ["60 focus", "60 rooms"],
        },
    },
    "hotel_reimbursement": {
        "question": "What are the hotel reimbursement limits for business travel?",
        "expected_facts": {
            "standard_rate": ["$200", "200 per night", "200/night"],
            "high_cost_rate": ["$275", "275"],
            "high_cost_cities": ["NYC", "San Francisco", "Chicago"],
            "submission_deadline": ["30 day", "30-day", "Sunset"],
        },
    },
    "ai_strategy": {
        "question": "What is the company's AI initiative and how much has been invested?",
        "expected_facts": {
            "project_name": ["Project Meridian", "Meridian"],
            "investment_amount": ["$2.1 million", "2.1M", "2,100,000"],
            "leverage_target": ["1.8x", "leverage ratio"],
            "training_budget": ["$450,000", "450"],
        },
    },
    "professional_growth": {
        "question": "What professional growth and learning benefits does Northbrook offer?",
        "expected_facts": {
            "dev_budget": ["$3,200", "3,200", "3200"],
            "curiosity_credit": ["Curiosity Credit", "$150"],
            "spark_sessions": ["Spark Session", "biweekly", "Thursday"],
            "conference_bonus": ["$800", "conference", "publication"],
        },
    },
}

In [ ]:
def evaluate(questions: dict, collection_name: str = "lab1",
             top_k: int = 3) -> list[dict]:
    """Run each question through ask() and check for expected facts.

    Args:
        questions: the EVAL_QUESTIONS dict
        collection_name: which collection to query
        top_k: chunks to retrieve per question

    Returns:
        list of dicts, each with: "key", "question", "facts_hit", "facts_total",
        "details" (dict of fact_name -> bool), "answer_preview" (first 200 chars)
    """
    results = []

    # -- Step 1 -------------------------------------------------------
    # Loop through each question in the questions dict.
    # Call ask() for each question.

    # -- Step 2 -------------------------------------------------------
    # For each expected fact, check if ANY of its keywords appear
    # in the answer (case-insensitive).
    # Hint: any(kw.lower() in answer.lower() for kw in keywords)

    # -- Step 3 -------------------------------------------------------
    # Build the result dict for this question and append to results.

    # YOUR CODE HERE

    return results

In [ ]:
# -- Run evaluation ----------------------------------------------------
results = evaluate(EVAL_QUESTIONS, LAB_COLLECTION)
print(f"{'Question':<22} {'Score':>8}")
print("-" * 32)
total_hits = 0
total_facts = 0
for r in results:
    total_hits += r["facts_hit"]
    total_facts += r["facts_total"]
    pct = r["facts_hit"] / r["facts_total"] * 100
    print(f"{r['key']:<22} {r['facts_hit']}/{r['facts_total']} ({pct:.0f}%)")
    for fact, hit in r["details"].items():
        print(f"  {'Y' if hit else '-'} {fact}")
print("-" * 32)
print(f"{'TOTAL':<22} {total_hits}/{total_facts} ({total_hits/total_facts*100:.0f}%)")

## Section 4: Analysis

Answer the following questions based on your evaluation results. Write 3-4 sentences for each.

**Q1:** Which question scored the worst? Look at the retrieved sources for that question — why do you think the pipeline struggled with it?

*Your answer:*



**Q2:** If you could change ONE parameter (`chunk_size`, `overlap`, or `top_k`) to improve your worst-scoring question, what would you change and why? You may re-run your pipeline to test this.

*Your answer:*



In [ ]:
# Optional: re-run with different parameters to test your hypothesis from Q2
